# Run on google colab only

This notebook is optimized for Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ln -sf '/content/drive/MyDrive/piano-chords/svm_gridsearch' '/content/svm_gridsearch'

Mounted at /content/drive


In [ ]:
!nvidia-smi

Fri Jun  5 14:55:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   51C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%load_ext cuml.accel
%load_ext cudf.pandas

In [ ]:
!pip install optuna -qqq && echo "successfully installed optuna"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 32.6 MB/s eta 0:00:00
successfully installed optuna


# Define constants

In [ ]:
from scipy.stats import loguniform

CLEAN_FEATURES_PATH = "./svm_gridsearch/opus-training.npz"
NOISY_FEATURES_PATH = "./svm_gridsearch/opus-evaluation.npz"
MODEL_SAVE_PATH = "./models"
STORAGE_NAME = "sqlite:///svm_gridsearch/svm_tuning.db"
STUDY_NAME = "RBF_SVM_Tuning-RandomizedSearchCV"
STUDY_OVERRIDE = True
RANDOM_STATE = 42
SVM_TEST_SIZE = 0.1
SVM_CV_FOLDS = 5
SVM_CV_N_ITER = 20
SEARCH_C = [0.1, 1.0, 10.0, 100.0, 1000.0]
SEARCH_GAMMA = [0.000001, 0.00001, 0.0001, 0.001] #  (1 / 40608) = 0.000025
SEARCH_N_TRIALS = 20 # continues from existing search

import os
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
SVM_MODEL_PATH = os.path.join(MODEL_SAVE_PATH, 'model.pkl')
SVM_SCALER_PATH = os.path.join(MODEL_SAVE_PATH, 'scaler.pkl')
SVM_ENCODER_PATH = os.path.join(MODEL_SAVE_PATH, 'label_encoder.pkl')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
import random
import numpy as np
import cupy as cp
import tensorflow as tf

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
cp.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


# Hyperparameter Tuning

## Load features

In [ ]:
import numpy as np
import cupy as cp

# Load the NPZ file using numpy to handle string labels
with np.load(CLEAN_FEATURES_PATH) as data:
    # Features are numerical, so we can move them to GPU immediately
    features = cp.array(data['features'])
    # Labels are strings; keep them as numpy for encoding in the next step
    labels = data['labels']

print(f"Loaded features shape: {features.shape}")
print(f"Loaded labels shape:   {labels.shape}")

Loaded features shape: (7200, 216, 188)
Loaded labels shape:   (7200,)


## Split data

In [ ]:
from cuml.model_selection import train_test_split
from cuml.preprocessing import LabelEncoder, StandardScaler

# Flatten 2D CQT features to 1D for SVM input
features_flat = features.reshape(features.shape[0], -1)

# Encode labels
svm_label_encoder = LabelEncoder()
svm_encoded_labels = svm_label_encoder.fit_transform(labels)

# Train/test split on the sampled data
X_svm_train, X_svm_test, y_svm_train, y_svm_test = train_test_split(
    features_flat,
    svm_encoded_labels,
    test_size=SVM_TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=svm_encoded_labels,
)

# Scale features
scaler = StandardScaler()

X_svm_train = scaler.fit_transform(X_svm_train)
X_svm_train = cp.asarray(X_svm_train)
y_svm_train = cp.asarray(y_svm_train)

X_svm_test = scaler.transform(X_svm_test)
X_svm_test = cp.asarray(X_svm_test)
y_svm_test = cp.asarray(y_svm_test)

print(f"Original dataset size: {len(features_flat)}")
print(f"X_svm_train shape:      {X_svm_train.shape}")
print(f"X_svm_test shape:       {X_svm_test.shape}")
print(f"Number of classes:      {len(svm_label_encoder.classes_)}")

Original dataset size: 7200
X_svm_train shape:      (6480, 40608)
X_svm_test shape:       (720, 40608)
Number of classes:      36


## GridSearchCV

In [ ]:
import optuna
import cupy as cp
import pandas as pd
from cuml.svm import SVC
from cuml.metrics import accuracy_score
from cuml.model_selection import StratifiedKFold
from cuml.preprocessing import StandardScaler
from datetime import datetime

def objective(trial):
    trial_start_time = datetime.now().replace(microsecond=0)

    # Pick from the discrete powers of 10
    c_val = trial.suggest_categorical("C", SEARCH_C)
    gamma_val = trial.suggest_categorical("gamma", SEARCH_GAMMA)

    # Instantiate the cuML model
    model = SVC(kernel='rbf', C=c_val, gamma=gamma_val, random_state=RANDOM_STATE)

    # Cross-Validation
    skf = StratifiedKFold(n_splits=SVM_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    print(f"Trial {trial.number} - C = {c_val} gamma = {gamma_val}")

    scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_svm_train, y_svm_train), 1):
        fold_start_time = datetime.now().replace(microsecond=0)
        X_tr, X_val = X_svm_train[train_idx], X_svm_train[val_idx]
        y_tr, y_val = y_svm_train[train_idx], y_svm_train[val_idx]

        print(f"  Trial {trial.number} - Fold {fold_idx}: Fitting...")
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        score = accuracy_score(y_val, preds)
        scores.append(score)

        fold_time = datetime.now().replace(microsecond=0) - fold_start_time
        print(f"  Trial {trial.number} - Fold {fold_idx}: Accuracy = {score:.4f} ({fold_time})")

    result = sum(scores) / len(scores)
    trial_time = datetime.now().replace(microsecond=0) - trial_start_time
    print(f"  Trial {trial.number} - C = {c_val} gamma = {gamma_val}: Result = {result} ({trial_time})")

    return result

if STUDY_OVERRIDE:
  try:
      optuna.delete_study(study_name=STUDY_NAME, storage=STORAGE_NAME)
      print(f"Override study '{STUDY_NAME}' to apply new parameters.")
  except KeyError:
      pass

# Create or load the Optuna study with RandomSampler
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_NAME,
    direction="maximize",
    load_if_exists=True,
    sampler=optuna.samplers.RandomSampler(seed=RANDOM_STATE)
)

print(f"Study {STUDY_NAME} is now persistent at {STORAGE_NAME}")

# Run the search
print("Optimizing objective...")
study.optimize(objective, n_trials=SEARCH_N_TRIALS, show_progress_bar=True, gc_after_trial=True)

# Tie-breaking for best parameters
df = study.trials_dataframe()
best_score = df['value'].max()
tied_trials = df[df['value'] == best_score]
best_configuration = tied_trials.sort_values(by=['params_C', 'params_gamma']).iloc[0]

print(f"Maximum CV Accuracy: {best_score:.6f}")
print(f"Optimal C: {best_configuration['params_C']:.6f}")
print(f"Optimal Gamma: {best_configuration['params_gamma']:.6f}")


[I 2026-06-05 14:59:23,689] A new study created in RDB with name: RBF_SVM_Tuning-RandomizedSearchCV


Study RBF_SVM_Tuning-RandomizedSearchCV is now persistent at sqlite:///svm_gridsearch/svm_tuning.db
Optimizing objective...


  0%|          | 0/20 [00:00<?, ?it/s]

Trial 0 - C = 1.0 gamma = 0.0001
  Trial 0 - Fold 1: Fitting...
  Trial 0 - Fold 1: Accuracy = 0.9961 (0:03:52)
  Trial 0 - Fold 2: Fitting...
  Trial 0 - Fold 2: Accuracy = 0.9969 (0:03:47)
  Trial 0 - Fold 3: Fitting...
  Trial 0 - Fold 3: Accuracy = 0.9969 (0:03:47)
  Trial 0 - Fold 4: Fitting...
  Trial 0 - Fold 4: Accuracy = 0.9985 (0:03:47)
  Trial 0 - Fold 5: Fitting...
  Trial 0 - Fold 5: Accuracy = 0.9977 (0:03:47)
  Trial 0 - C = 1.0 gamma = 0.0001: Result = 0.9972222222222221 (0:19:03)
[I 2026-06-05 15:18:27,531] Trial 0 finished with value: 0.9972222222222221 and parameters: {'C': 1.0, 'gamma': 0.0001}. Best is trial 0 with value: 0.9972222222222221.
Trial 1 - C = 10.0 gamma = 0.001
  Trial 1 - Fold 1: Fitting...
  Trial 1 - Fold 1: Accuracy = 0.0324 (0:03:47)
  Trial 1 - Fold 2: Fitting...
  Trial 1 - Fold 2: Accuracy = 0.0316 (0:03:48)
  Trial 1 - Fold 3: Fitting...
  Trial 1 - Fold 3: Accuracy = 0.0324 (0:03:49)
  Trial 1 - Fold 4: Fitting...
  Trial 1 - Fold 4: Accuracy

KeyboardInterrupt: 

## Checkpoint

In [ ]:
print(f"Maximum CV Accuracy: {best_score}")
print(f"Optimal C (Tie-broken): {best_configuration['params_C']}")
print(f"Optimal Gamma (Tie-broken): {best_configuration['params_gamma']}")

NameError: name 'best_score' is not defined